# UNSW-NB15 Linear Transformer HLS CSYNTH

This notebook runs or verifies Vitis HLS `csynth_design`, parses timing/resource reports,
and decides whether the float binary anomaly-detection accelerator can proceed toward
PYNQ-Z2 integration.

It does not export IP, run Vivado, generate a bitstream, or access a PYNQ board.

## 2. Import dependencies

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

print("Python:", sys.version.split()[0])

Python: 3.12.7


## 3. Path configuration

In [2]:
PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_unsw_linear_csynth import (
    CSIM_REPORT_PATH,
    CSYNTH_RESULTS_DIR,
    HLS_PROJECT_DIR,
    PYNQ_Z2_CAPACITY,
    REQUIRED_PROJECT_FILES,
    RUN_CSYNTH_TCL,
    TARGET_CLOCK_NS,
    TARGET_PART,
    TOP_FUNCTION,
    check_csim_passed,
    check_project_files,
    run_pipeline,
)

print("HLS project:", HLS_PROJECT_DIR)
print("CSYNTH results:", CSYNTH_RESULTS_DIR)
print("Target part:", TARGET_PART)
print("Target clock:", TARGET_CLOCK_NS, "ns")

HLS project: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls
CSYNTH results: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/hls/csynth
Target part: xc7z020clg400-1
Target clock: 10.0 ns


## 4. Check HLS project files

In [3]:
project_files = check_project_files()
print("Required HLS project files:")
for path in project_files:
    print(" -", path)

Required HLS project files:
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/unsw_linear_transformer.h
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/unsw_linear_transformer.cpp
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/weights.h
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/test_vectors.h
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/testbench.cpp
 - /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/run_csynth.tcl


## 5. Require a passing CSIM report

In [4]:
csim_report = check_csim_passed()
print(json.dumps(csim_report, indent=2))
assert csim_report["csim_passed"] is True

{
  "vitis_hls_available": true,
  "vitis_hls_command": "/tools/Xilinx/Vitis_HLS/2022.2/bin/vitis_hls",
  "csim_executed": true,
  "csim_passed": true,
  "return_code": 0,
  "metrics": {
    "max_abs_error": 1.907348633e-06,
    "mean_abs_error": 4.954636097e-07,
    "prediction_match_rate": 1.0
  },
  "reason": "CSIM PASS"
}


## 6. Check `run_csynth.tcl`

In [5]:
tcl_text = RUN_CSYNTH_TCL.read_text(encoding="utf-8")
print(tcl_text)

assert "set_top unsw_linear_transformer" in tcl_text
assert "set_part xc7z020clg400-1" in tcl_text
assert "create_clock -period 10 -name default" in tcl_text
assert "csynth_design" in tcl_text
assert "export_design" not in tcl_text

open_project -reset unsw_linear_transformer_prj
set_top unsw_linear_transformer
add_files src/unsw_linear_transformer.cpp
add_files src/unsw_linear_transformer.h
add_files src/weights.h
add_files -tb src/testbench.cpp
add_files -tb src/test_vectors.h
open_solution "solution1"
set_part xc7z020clg400-1
create_clock -period 10 -name default
csynth_design
exit



## 7. Run or verify Vitis HLS `csynth_design`

In [6]:
result = run_pipeline(force=False)
csynth_status = result["csynth_status"]
metrics = result["metrics"]
log_analysis = result["log_analysis"]

print(json.dumps(csynth_status, indent=2))
if not csynth_status["csynth_succeeded"]:
    print("Last 80 log lines:")
    print("\n".join(log_analysis["last_80_lines"]))
    raise RuntimeError("csynth_design failed; do not continue to hardware integration")

{
  "vitis_hls_available": true,
  "vitis_hls_command": "/tools/Xilinx/Vitis_HLS/2022.2/bin/vitis_hls",
  "csynth_executed": true,
  "csynth_reused": true,
  "return_code": 0,
  "report_path": "/home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/unsw_linear_transformer_prj/solution1/syn/report/csynth.rpt",
  "reason": "Current csynth report reused",
  "csynth_succeeded": true
}


## 8. Parse `csynth.rpt`

In [7]:
report_path = Path(metrics["report_path"])
copied_report_path = Path(metrics["copied_report_path"])

print("Original report:", report_path)
print("Copied report:", copied_report_path)
print("Metrics source:", metrics["metrics_source"])
print("Report warnings:", metrics["report_warning_count"])
print("Report errors:", metrics["report_error_count"])
print("Report size:", copied_report_path.stat().st_size, "bytes")

Original report: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/unsw_linear_transformer_prj/solution1/syn/report/csynth.rpt
Copied report: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/hls/csynth/unsw_linear_transformer_csynth.rpt
Metrics source: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/unsw_linear_transformer_prj/solution1/syn/report/csynth.xml
Report warnings: 0
Report errors: 0
Report size: 60847 bytes


## 9. Parse the Vitis HLS log

In [8]:
print("Log warning count:", log_analysis["warning_count"])
print("Log error count:", log_analysis["error_count"])
print("Warnings (first 10):")
for line in log_analysis["warning_lines"][:10]:
    print(" -", line)

assert log_analysis["error_count"] == 0

Log warning count: 37
Log error count: 0
Warnings (first 10):
 - WARNING: [HLS 200-960] Cannot flatten loop 'VITIS_LOOP_127_12' (src/unsw_linear_transformer.cpp:128:15) in function 'unsw_linear_transformer' more than one sub loop.
 - WARNING: [HLS 200-885] The II Violation in module 'unsw_linear_transformer_Pipeline_VITIS_LOOP_71_1_VITIS_LOOP_72_2' (loop 'VITIS_LOOP_71_1_VITIS_LOOP_72_2'): Unable to schedule 'load' operation ('input_r_load_1', src/unsw_linear_transformer.cpp:71) on array 'input_r' due to limited memory ports (II = 1). Please consider using a memory core with more ports or partitioning the array 'input_r'.
 - WARNING: [HLS 200-885] The II Violation in module 'unsw_linear_transformer_Pipeline_VITIS_LOOP_71_1_VITIS_LOOP_72_2' (loop 'VITIS_LOOP_71_1_VITIS_LOOP_72_2'): Unable to schedule 'load' operation ('input_r_load_3', src/unsw_linear_transformer.cpp:71) on array 'input_r' due to limited memory ports (II = 2). Please consider using a memory core with more ports or parti

## 10. Summarize timing, latency, interval, and resources

In [9]:
summary_fields = {
    "solution": metrics["solution_name"],
    "top_function": metrics["top_function"],
    "target_part": metrics["target_part"],
    "target_clock_ns": metrics["target_clock_period_ns"],
    "estimated_clock_ns": metrics["estimated_clock_period_ns"],
    "latency_min_cycles": metrics["latency_min_cycles"],
    "latency_max_cycles": metrics["latency_max_cycles"],
    "interval_min_cycles": metrics["interval_min_cycles"],
    "interval_max_cycles": metrics["interval_max_cycles"],
    "BRAM_18K": metrics["BRAM_18K"],
    "DSP48E": metrics["DSP48E"],
    "FF": metrics["FF"],
    "LUT": metrics["LUT"],
    "URAM": metrics["URAM"],
}
display(pd.DataFrame([summary_fields]).T.rename(columns={0: "value"}))

,value
solution,solution1
top_function,unsw_linear_transformer
target_part,xc7z020-clg400-1
target_clock_ns,10.0
estimated_clock_ns,8.567
latency_min_cycles,7869
latency_max_cycles,7869
interval_min_cycles,7870
interval_max_cycles,7870
BRAM_18K,176


## 11. Assess PYNQ-Z2 feasibility

In [10]:
resource_rows = []
for resource, detail in metrics["resource_utilization"].items():
    resource_rows.append(
        {
            "resource": resource,
            "used": detail["used"],
            "capacity": detail["capacity"],
            "utilization_percent": detail["utilization_percent"],
            "over_capacity": detail["utilization_percent"] > 100.0,
        }
    )
resource_table = pd.DataFrame(resource_rows)
display(resource_table)

print("Timing violation:", metrics["timing_violation"])
print("Resource over limit:", metrics["resource_over_limit"])
print("DSP or LUT above 80%:", metrics["dsp_or_lut_over_80_percent"])
print("Can continue to export IP/Vivado:", metrics["can_continue_to_export_ip_vivado"])
print("Recommendation:", metrics["recommendation"])

if metrics["resource_over_limit"]:
    print(
        "The float design cannot be used for board anomaly-detection metrics yet. "
        "Apply ap_fixed quantization and/or structural optimization, then repeat CSIM and CSYNTH."
    )

,resource,used,capacity,utilization_percent,over_capacity
0,BRAM_18K,176,280,62.857143,False
1,DSP48E,258,220,117.272727,True
2,FF,71828,106400,67.507519,False
3,LUT,65546,53200,123.206767,True


Timing violation: False
Resource over limit: True
DSP or LUT above 80%: True
Can continue to export IP/Vivado: False
Recommendation: Stop: the float design exceeds PYNQ-Z2 resources. Convert to ap_fixed and/or optimize the structure before board accuracy testing.
The float design cannot be used for board anomaly-detection metrics yet. Apply ap_fixed quantization and/or structural optimization, then repeat CSIM and CSYNTH.


## 12. Verify saved CSYNTH reports

In [11]:
required_outputs = [
    CSYNTH_RESULTS_DIR / "hls_csynth_output.txt",
    CSYNTH_RESULTS_DIR / "unsw_linear_transformer_csynth.rpt",
    CSYNTH_RESULTS_DIR / "csynth_metrics.json",
    CSYNTH_RESULTS_DIR / "csynth_metrics.csv",
    CSYNTH_RESULTS_DIR / "csynth_summary.md",
    CSYNTH_RESULTS_DIR / "generated_file_list.txt",
]
for path in required_outputs:
    if not path.is_file():
        raise FileNotFoundError(f"Missing expected output: {path}")

print("CSYNTH summary:", CSYNTH_RESULTS_DIR / "csynth_summary.md")
print("Can enter export IP/Vivado:", metrics["can_continue_to_export_ip_vivado"])
print("Saved outputs:")
for path in required_outputs:
    print(f" - {path.name}: {path.stat().st_size} bytes")

CSYNTH summary: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/hls/csynth/csynth_summary.md
Can enter export IP/Vivado: False
Saved outputs:
 - hls_csynth_output.txt: 143058 bytes
 - unsw_linear_transformer_csynth.rpt: 60847 bytes
 - csynth_metrics.json: 2033 bytes
 - csynth_metrics.csv: 1522 bytes
 - csynth_summary.md: 1917 bytes
 - generated_file_list.txt: 378 bytes
